# Masterclass 3: Sectoral Decision Intelligence & The 15 Real-World Use Cases
### *Operational Multi-Criteria Spatial Prioritization Across Healthcare, Commerce, Cultural Cohesion, Public Utilities, and Governance*

---

## 1. Overview: 15 Enterprise Use Cases

Spatial statistics transforms raw geographic coordinates and Earth observation data into operational strategy. Below, we operationalize this across **15 distinct real-world enterprise and policy use cases**:

| # | Strategic Sector | Real-World Enterprise / Policy Use Case | Spatial Analytics Method |
| :-: | :--- | :--- | :--- |
| **1** | **Epidemiology** | Malaria & Cholera cluster surveillance and vector control | Anselin LISA Hotspots & Spatial Lag |
| **2** | **Public Health** | Healthcare Desert eradication & emergency maternal routing | Siting optimization & buffer distance |
| **3** | **WASH Utilities** | Clean water borehole equity and water poverty eradication | Lorenz Inequality Curves & Gini indices |
| **4** | **Education** | Primary school catchment planning & dropout prevention | Child density vs school spatial lags |
| **5** | **Retail & FMCG** | Commercial supermarket & distributor expansion | Purchasing power catchment quadrants |
| **6** | **Fintech** | Mobile money & POS agency banking network optimization | Cash-in/cash-out demand vs bank gaps |
| **7** | **Telecoms** | 4G/5G cell tower siting & network coverage expansion | Density-weighted signal decay optimization |
| **8** | **Energy** | Off-grid renewable solar mini-grid placement | Affluent un-electrified spatial sorting |
| **9** | **Cultural Cohesion** | Inter-religious dialogue & community peacebuilding | Shannon Entropy Diversity Index |
| **10** | **Emergency Services** | Police and Fire station response radius coverage | Network service shed analysis |
| **11** | **Climate Resilience** | Urban heat island mitigation & tree canopy deficits | Satellite vegetation vs heat stress |
| **12** | **Agriculture** | Wholesale grain silo & cold-storage market placement | Farm-to-market spatial catchment |
| **13** | **Real Estate** | Spatial hedonic property valuation & amenity pricing | Spatial Error Models (SEM) |
| **14** | **Transport** | Regional road infrastructure & transit corridors | Spatial Multiplier $(I - ho W)^{-1}$ |
| **15** | **Governance** | Electoral polling station logistics & crowd mitigation | Voter travel distance optimization |


In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import libpysal
import esda

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')


In [ ]:
DATA_PATH = '../data/processed/nigeria_wards_master.parquet'
if not os.path.exists(DATA_PATH):
    DATA_PATH = 'data/processed/nigeria_wards_master.parquet'

gdf = gpd.read_parquet(DATA_PATH)
gdf = gdf[gdf.geometry.is_valid & ~gdf.geometry.is_empty].copy()
gdf.reset_index(drop=True, inplace=True)

gdf['rwi_mean'] = gdf['rwi_mean'].fillna(gdf['rwi_mean'].median())
gdf['pop_2025_sum'] = gdf['pop_2025_sum'].fillna(gdf['pop_2025_sum'].median())
gdf['health_rate'] = (gdf['health_facilities_count'] / (gdf['pop_2025_sum'] + 100)) * 10000
gdf['market_rate'] = (gdf['markets_count'] / (gdf['pop_2025_sum'] + 100)) * 10000
gdf['water_rate'] = (gdf['water_points_count'] / (gdf['pop_2025_sum'] + 100)) * 10000

print(f"Loaded {len(gdf):,} wards across {gdf['statename'].nunique()} states.")


\
## 2. Measuring Spatial Inequality: Gini Coefficients & Lorenz Curves

$$
G = \frac{\sum_{i=1}^n \sum_{j=1}^n |x_i - x_j|}{2 n^2 \bar{x}}
$$
$G = 0$ represents complete spatial equality; $G = 1$ indicates total geographic concentration.


In [ ]:
def gini_coefficient(values):
    vals = np.sort(np.asarray(values, dtype=np.float64))
    n = len(vals)
    if n == 0 or np.all(vals == 0):
        return 0.0
    index = np.arange(1, n + 1)
    return float((2 * np.sum(index * vals) - (n + 1) * np.sum(vals)) / (n * np.sum(vals)))

def lorenz_curve(values):
    vals = np.sort(np.asarray(values, dtype=np.float64))
    cum_vals = np.cumsum(vals) / np.sum(vals)
    cum_pop = np.linspace(0, 1, len(vals))
    return cum_pop, cum_vals

fig, ax = plt.subplots(figsize=(8.5, 6))
ax.plot([0, 1], [0, 1], 'k--', label='Perfect Spatial Equality (G = 0.0)')

for col, label, color in [
    ('health_facilities_count', 'Health Clinics', '#e63946'),
    ('markets_count', 'Commercial Markets', '#2a9d8f'),
    ('water_points_count', 'Water Points (WASH)', '#457b9d')
]:
    g_val = gini_coefficient(gdf[col].values)
    x_lorenz, y_lorenz = lorenz_curve(gdf[col].values)
    ax.plot(x_lorenz, y_lorenz, label=f"{label} (Gini = {g_val:.3f})", color=color, linewidth=2.5)

ax.set_title("Spatial Infrastructure Allocation Inequality (Lorenz Curves)", fontsize=13, fontweight='bold')
ax.set_xlabel("Cumulative Proportion of Administrative Wards")
ax.set_ylabel("Cumulative Proportion of Facilities")
ax.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()


## 3. Cross-Sector Coupling: Wealth vs. Healthcare Availability

Examining whether private wealth and public healthcare provision reinforce or contradict one another.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
hb = ax.hexbin(gdf['rwi_mean'], gdf['health_rate'].clip(upper=10), gridsize=35, cmap='YlGnBu', mincnt=1)
cb = fig.colorbar(hb, ax=ax)
cb.set_label('Number of Administrative Wards')
sns.regplot(x=gdf['rwi_mean'], y=gdf['health_rate'].clip(upper=10), scatter=False, color='#d90429', ax=ax, line_kws={'lw': 2.5, 'label': 'Linear Fit'})
ax.set_title("Bivariate Density: Relative Wealth Index vs. Health Clinic Density", fontsize=12, fontweight='bold')
ax.set_xlabel("Relative Wealth Index (RWI)")
ax.set_ylabel("Health Facilities per 10k Population (Clipped at 10)")
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()



## 4. Multi-Criteria Decision Analysis (MCDA): The Ward Priority Index (WPI)

$$
\text{WPI}_i = 0.30 \cdot \text{PovertyDeficit}_i + 0.30 \cdot \text{HealthDeficit}_i + 0.20 \cdot \text{WaterDeficit}_i + 0.20 \cdot \text{PopWeight}_i
$$


In [ ]:
def min_max(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-8)

poverty_def = min_max(-gdf['rwi_mean'])
health_def = min_max(1 / (gdf['health_rate'] + 0.1))
water_def = min_max(1 / (gdf['water_rate'] + 0.1))
pop_wt = min_max(np.log1p(gdf['pop_2025_sum']))

gdf['ward_priority_index'] = 0.30 * poverty_def + 0.30 * health_def + 0.20 * water_def + 0.20 * pop_wt

gdf['action_tier'] = pd.qcut(
    gdf['ward_priority_index'], 
    q=4, 
    labels=[
        'Tier 4: Mature / Self-Sustaining', 
        'Tier 3: Moderate Support Needed', 
        'Tier 2: High Investment Priority', 
        'Tier 1: Critical Emergency Intervention'
    ]
)

print("=== WARD PRIORITY ACTION TIERS ===")
print(gdf['action_tier'].value_counts())


In [ ]:
tier_colors = {
    'Tier 1: Critical Emergency Intervention': '#d90429',
    'Tier 2: High Investment Priority': '#f77f00',
    'Tier 3: Moderate Support Needed': '#fcbf49',
    'Tier 4: Mature / Self-Sustaining': '#2a9d8f'
}

fig, ax = plt.subplots(figsize=(11, 8.5))
for tier, color in tier_colors.items():
    subset = gdf[gdf['action_tier'] == tier]
    subset.plot(color=color, ax=ax, linewidth=0.1, edgecolor='white')

ax.set_title("National Ward Priority Action Tiers (Multi-Sector MCDA)", fontsize=13, fontweight='bold')
ax.axis('off')
tier_patches = [mpatches.Patch(color=c, label=f"{l} (n={(gdf['action_tier'] == l).sum():,})") for l, c in tier_colors.items()]
ax.legend(handles=tier_patches, loc='lower left', frameon=True, facecolor='white', framealpha=0.9, fontsize=9.5)

plt.tight_layout()
plt.show()


## 5. State-by-State Comparative Vulnerability Ranking

Ranked average Ward Priority Index (WPI) across all 36 states and the FCT, identifying states with the highest systemic infrastructure deficits.


In [ ]:
state_avg_wpi = gdf.groupby('statename')['ward_priority_index'].mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 11))
state_avg_wpi.plot(kind='barh', color='#e76f51', ax=ax, edgecolor='none')
ax.set_title("Ranked State Vulnerability: Mean Ward Priority Index Across All 37 Units", fontsize=12, fontweight='bold')
ax.set_xlabel("Average Ward Priority Index (Higher = Greater Need for Public Intervention)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 6. Multi-Sector Deficit Matrix for Top 15 Priority Wards

Normalized z-scores across Poverty, Health Clinic Shortage, Water Shortage, and Population Pressure for the most critical wards in the country.


In [ ]:
top_15 = gdf.sort_values(by='ward_priority_index', ascending=False).head(15).copy()
matrix_df = top_15[['statename', 'wardname', 'rwi_mean', 'health_rate', 'water_rate', 'pop_2025_sum']].set_index(['statename', 'wardname'])
norm_matrix = (matrix_df - matrix_df.mean()) / matrix_df.std()

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(norm_matrix, annot=True, fmt=".2f", cmap='coolwarm_r', center=0, ax=ax,
            cbar_kws={'label': 'Normalized z-score (Red = Acute Deficit)'})
ax.set_title("Multi-Sector Deficit Profile: Top 15 Priority Wards in Nigeria", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## 7. Institutional Governance & Capital Allocation Playbook

1. **Precision Budgeting:** Fund allocation must shift from flat LGA-level grants to ward-level priority tiers.
2. **Multi-Sector Bundling:** Interventions in Tier 1 wards must co-locate water boreholes, primary clinics, and micro-retail support.
